# Layer 2 — Data Understanding & EDA
In this notebook we inspect the Telco Churn dataset to uncover data types, hidden missing values, target imbalance, and potential leakage.

In [5]:
import pandas as pd

df = pd.read_csv('../data/raw/Telco-Customer-Churn.csv')
df.shape

(7043, 21)

### 1. Missing Data Percentage Check
Before modeling, check what percentage of each column is missing/blank.

In [6]:
# Coerce TotalCharges to numeric so blanks become NaN
df_clean = df.copy()
df_clean['TotalCharges'] = pd.to_numeric(df_clean['TotalCharges'], errors='coerce')

missing_pct = (df_clean.isnull().sum() / len(df_clean)) * 100
print("Columns with missing values (%):")
print(missing_pct[missing_pct > 0])

Columns with missing values (%):
TotalCharges    0.156183
dtype: float64


Notice: Only `TotalCharges` is missing, and it is only **0.15%** (11 rows). No feature has excessive missing data.

### 2. The `TotalCharges` Anomaly
Notice why `TotalCharges` was loaded as an `object` / `string` rather than a `float`.

In [7]:
# Find rows where TotalCharges is just spaces or non-numeric
blank_rows = df[df['TotalCharges'].str.strip() == '']
print(f"Found {len(blank_rows)} rows with empty string in TotalCharges.")
blank_rows[['customerID', 'tenure', 'MonthlyCharges', 'TotalCharges', 'Churn']]

Found 11 rows with empty string in TotalCharges.


,customerID,tenure,MonthlyCharges,TotalCharges,Churn
488,4472-LVYGI,0,52.55,,No
753,3115-CZMZD,0,20.25,,No
936,5709-LVOEQ,0,80.85,,No
1082,4367-NUYAO,0,25.75,,No
1340,1371-DWPAZ,0,56.05,,No
3331,7644-OMVMY,0,19.85,,No
3826,3213-VVOLG,0,25.35,,No
4380,2520-SGTTA,0,20.00,,No
5218,2923-ARZLG,0,19.70,,No
6670,4075-WKNIU,0,73.35,,No


### 3. Class Imbalance
What proportion of customers churned?

In [8]:
churn_counts = df['Churn'].value_counts(normalize=True)
print(churn_counts)
print(f"Churn ratio: {churn_counts['No'] / churn_counts['Yes']:.1f} to 1")

Churn
No     0.73463
Yes    0.26537
Name: proportion, dtype: float64
Churn ratio: 2.8 to 1


### 4. Feature Breakdown & Stratified Split

In [9]:
from sklearn.model_selection import train_test_split

y = (df['Churn'] == 'Yes').astype(int)
X = df.drop(columns=['customerID', 'Churn'])

# 70% Train, 15% Val, 15% Test
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp)

print(f"Train: {X_train.shape[0]} rows, churn: {y_train.mean():.4f}")
print(f"Val:   {X_val.shape[0]} rows, churn: {y_val.mean():.4f}")
print(f"Test:  {X_test.shape[0]} rows, churn: {y_test.mean():.4f}")

Train: 4930 rows, churn: 0.2653
Val:   1056 rows, churn: 0.2652
Test:  1057 rows, churn: 0.2658
